**Nombre de Proyecto: Proyecto EdTech Learning Analytics Parte 1**

**Numero de grupo: 2**

**Integrante a cargo: Luis Alonso Hernandez Mundo**

**Co-Autora: Isela Alexandra Quintanilla Cruz**

**Carnet: 039025**

**Tipo de libro: Libro python para evaluacion y conexion de fuentes de datos**

**Fecha de creacion: 15/09/2026**

**Version de libro: 1.1**

In [ ]:
# Creacion de carpeteo
!mkdir -p datos/crudos
!mkdir -p datos/procesados
!mkdir -p datos/externos

print("Carpetas creadas!!!!")

Carpetas creadas!!!!


In [ ]:
# Importacion de librerias
import pandas as pd
import sqlite3
import json
from pathlib import Path

print("Librerias importadas!!!!")

Librerias importadas!!!!


In [ ]:
# Definicion de rutas de trabajo
BASE_DIR = Path("/content/")
DATA_RAW = BASE_DIR /"datos"/"crudos"
DATA_PROCESSED = BASE_DIR /"datos"/"procesados"
DATA_EXTERNAL = BASE_DIR /"datos"/"externos"

print(f"Datos crudos: {DATA_RAW}")
print(f"Datos procesados: {DATA_PROCESSED}")
print(f"Datos externos: {DATA_EXTERNAL}")

Datos crudos: /content/datos/crudos
Datos procesados: /content/datos/procesados
Datos externos: /content/datos/externos


In [ ]:
# Carga de primera fuente de datos: Archivo CSV
df_Alumnado_UEScsv = pd.read_csv("/content/datos/crudos/Alumnado_UES.csv")
print("\nLeyendo el archivo CSV")
print(df_Alumnado_UEScsv.head(5))


Leyendo el archivo CSV
    carnet          nombres        apellidos fecha_nacimiento  edad      sexo  \
0  RR19001   Carlos Ricardo  Ramírez Ramírez       1999-02-22    27         M   
1  RF24002  DANIELA CLAUDIA    Romero Flores       2006-03-23    20  FEMENINO   
2  FG23003   mariana fatima  FLORES GONZÁLEZ       2004-10-01    21  FEMENINO   
3  RL21004     roberto luis    Ramírez López       2002-08-21    24         M   
4  PM21005     Hector Jorge    Pérez Morales       2002-11-24    23         m   

                    facultad                               carrera  \
0       Ciencias Agronómicas                 Ingeniería Agronómica   
1       Ciencias Agronómicas  Licenciatura en Medicina Veterinaria   
2  Ingeniería y Arquitectura                  Ingeniería Eléctrica   
3       Ciencias Agronómicas  Licenciatura en Medicina Veterinaria   
4       Ciencias Agronómicas                 Ingeniería Agronómica   

   anio_ingreso  ciclo_actual       turno  municipio_residencia    z

In [ ]:
# Carga de Segunda fuente de datos: Archivo SQL
ruta_sql = "/content/datos/crudos/Rendimiento_academicoUES.sql"
ruta_db = "/content/datos/crudos/Rendimiento_academicoUES.db"

conn = sqlite3.connect(ruta_db)
cursor = conn.cursor()

# Nota del desarrollador: Esto lo hice porque mi archivo de base es .sql y no .db asi que esto es para transformar
# de .sql a .db
# 1. Leer el archivo con codificación latin-1
with open(ruta_sql, 'r', encoding='latin-1') as f:
    lineas = f.readlines()

# 2. Filtrar líneas incompatibles con SQLite (DATABASE, USE, etc.)
lineas_limpias = []
for linea in lineas:
    linea_upper = linea.strip().upper()
    if linea_upper.startswith("CREATE DATABASE") or \
       linea_upper.startswith("USE ") or \
       linea_upper.startswith("SET ") or \
       linea_upper.startswith("LOCK TABLES") or \
       linea_upper.startswith("UNLOCK TABLES"):
        continue
    lineas_limpias.append(linea)

sql_script = "".join(lineas_limpias)

# 3. Ejecutar el script filtrado
cursor.executescript(sql_script)
conn.commit()

print("¡Base de datos cargada exitosamente!")

¡Base de datos cargada exitosamente!


In [ ]:
# Contar el número de filas guardadas en la tabla
cursor.execute("SELECT COUNT(*) FROM Rendimiento_academico;")
total_filas = cursor.fetchone()[0]

print(f"Total de registros cargados en la base de datos: {total_filas}")

Total de registros cargados en la base de datos: 1000


In [ ]:
import pandas as pd

# Ver qué tablas se crearon realmente
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tablas creadas:", cursor.fetchall())

# Cargar el DataFrame
df_BaseRendimientoAcademicodb = pd.read_sql_query("SELECT * FROM Rendimiento_academico", conn)
df_BaseRendimientoAcademicodb.head(5)

Tablas creadas: [('Rendimiento_academico',)]


,carnet,ciclo,materia,nota_teoria,nota_laboratorio,nota_final,faltas_consecutivas,materia_reprobada,observacion_docente,tareas_entregadas,tareas_tarde,participacion_nivel,nivel_riesgo,probabilidad_desercion
0,RR19001,10,Ingles,7.89,4.41,6.15,5,0,Buen desempeno,3,2,Alta,Bajo,0.5760
1,RF24002,6,Estadistica,6.43,7.55,6.99,0,0,Debe mejorar,2,1,Alta,Alto,0.1749
2,FG23003,7,Fisica I,9.40,8.64,9.02,5,0,Requiere seguimiento,5,1,Baja,Bajo,0.2405
3,RL21004,10,Ingles,6.04,7.69,6.86,5,0,Buen desempeno,9,3,Alta,Medio,0.1089
4,PM21005,10,Programacion I,9.39,4.50,6.94,5,0,Sin observaciones,8,0,Baja,Medio,0.5903


In [ ]:
#Carga de tercera fuente de datos: Archivo JSON
df_BitacoraEstudiantesjson = pd.read_json("/content/datos/externos/bitacora_estudiantes_ues.json")
print("Bitacora como DataFrame")
print(df_BitacoraEstudiantesjson.head(5))

Bitacora como DataFrame
    carnet                              historial_cualitativo
0  RR19001  {'total_observaciones': 2, 'requiere_seguimien...
1  RF24002  {'total_observaciones': 2, 'requiere_seguimien...
2  FG23003  {'total_observaciones': 1, 'requiere_seguimien...
3  RL21004  {'total_observaciones': 2, 'requiere_seguimien...
4  PM21005  {'total_observaciones': 1, 'requiere_seguimien...


In [ ]:
# Exploracion de la estructura de cada fuente (columnas, tipo, volumen)

# Exploracion del archivo CSV
print("=" * 31)
print("1. ESTRUCTURA: Alumnado_UES.csv")
print("=" * 31)
print(f"Volumen (Filas × Columnas): {df_Alumnado_UEScsv.shape}")
print("\nTipos de datos por columna:")
print(df_Alumnado_UEScsv.dtypes)
print("\nVista previa (primeras 3 filas):")
print(df_Alumnado_UEScsv.head(3))

1. ESTRUCTURA: Alumnado_UES.csv
Volumen (Filas × Columnas): (1000, 19)

Tipos de datos por columna:
carnet                   object
nombres                  object
apellidos                object
fecha_nacimiento         object
edad                      int64
sexo                     object
facultad                 object
carrera                  object
anio_ingreso              int64
ciclo_actual              int64
turno                    object
municipio_residencia     object
zona                     object
telefono                  int64
correo_institucional     object
acceso_internet_casa     object
trabaja                  object
horas_trabajo_semana    float64
beca                     object
dtype: object

Vista previa (primeras 3 filas):
    carnet          nombres        apellidos fecha_nacimiento  edad      sexo  \
0  RR19001   Carlos Ricardo  Ramírez Ramírez       1999-02-22    27         M   
1  RF24002  DANIELA CLAUDIA    Romero Flores       2006-03-23    20  FEMENINO   
2

In [ ]:
# Exploracion de la base de datos db
print("\n" + "=" * 73)
print("2. ESTRUCTURA: Rendimiento_academicoUES.db (Tabla: Rendimiento_academico)")
print("=" * 73)
print(f"Volumen (Filas × Columnas): {df_BaseRendimientoAcademicodb.shape}")
print("\nTipos de datos por columna:")
print(df_BaseRendimientoAcademicodb.dtypes)
print("\nVista previa (primeras 3 filas):")
print(df_BaseRendimientoAcademicodb.head(3))


2. ESTRUCTURA: Rendimiento_academicoUES.db (Tabla: Rendimiento_academico)
Volumen (Filas × Columnas): (1000, 14)

Tipos de datos por columna:
carnet                     object
ciclo                      object
materia                    object
nota_teoria               float64
nota_laboratorio          float64
nota_final                float64
faltas_consecutivas         int64
materia_reprobada           int64
observacion_docente        object
tareas_entregadas           int64
tareas_tarde                int64
participacion_nivel        object
nivel_riesgo               object
probabilidad_desercion    float64
dtype: object

Vista previa (primeras 3 filas):
    carnet ciclo      materia  nota_teoria  nota_laboratorio  nota_final  \
0  RR19001    10       Ingles         7.89              4.41        6.15   
1  RF24002     6  Estadistica         6.43              7.55        6.99   
2  FG23003     7     Fisica I         9.40              8.64        9.02   

   faltas_consecutivas  mate

In [ ]:
# Exploracion del archivo JSON
print("\n" + "=" *73)
print("3. ESTRUCTURA: bitacora_estudiantes_ues.json (df_BitacoraEstudiantesjson)")
print("=" * 73)
print(f"Volumen (Filas × Columnas): {df_BitacoraEstudiantesjson.shape}")
print("\nTipos de datos por columna:")
print(df_BitacoraEstudiantesjson.dtypes)
print("\nVista previa (primeras 3 filas):")
print(df_BitacoraEstudiantesjson.head(3))


3. ESTRUCTURA: bitacora_estudiantes_ues.json (df_BitacoraEstudiantesjson)
Volumen (Filas × Columnas): (1000, 2)

Tipos de datos por columna:
carnet                   object
historial_cualitativo    object
dtype: object

Vista previa (primeras 3 filas):
    carnet                              historial_cualitativo
0  RR19001  {'total_observaciones': 2, 'requiere_seguimien...
1  RF24002  {'total_observaciones': 2, 'requiere_seguimien...
2  FG23003  {'total_observaciones': 1, 'requiere_seguimien...


In [ ]:
# Consolidacion de datos en unico Dataframe de Pandas
# Primero hare una limpieza de llave primaria: esto es para asegurar que no haya espacios en blanco en la columna 'carnet'
df_Alumnado_UEScsv['carnet'] = df_Alumnado_UEScsv['carnet'].astype(str).str.strip()
df_BaseRendimientoAcademicodb['carnet'] = df_BaseRendimientoAcademicodb['carnet'].astype(str).str.strip()
df_BitacoraEstudiantesjson['carnet'] = df_BitacoraEstudiantesjson['carnet'].astype(str).str.strip()

# Luego aqui hice una union consecutiva (Inner Join) usando la columna 'carnet'
# Primero uni el CSV (Alumnado) con la Base de datos de (Rendimiento)
df_consolidado = pd.merge(
    df_Alumnado_UEScsv,
    df_BaseRendimientoAcademicodb,
    on='carnet',
    how='inner'
)

# Luego uni el resultado anterior con el JSON (Bitacora)
df_consolidado = pd.merge(
    df_consolidado,
    df_BitacoraEstudiantesjson,
    on='carnet',
    how='inner'
)

# Aqui mando a llamar la verificacion de la consolidacion
print("=========================================")
print(f"Estructura final: {df_consolidado.shape[0]} filas × {df_consolidado.shape[1]} columnas")
print("=========================================\n")

# Y finalmente hago un Muestreo de las primeras filas del DataFrame unificado
df_consolidado.head(5)

Estructura final: 1000 filas × 33 columnas



,carnet,nombres,apellidos,fecha_nacimiento,edad,sexo,facultad,carrera,anio_ingreso,ciclo_actual,...,nota_final,faltas_consecutivas,materia_reprobada,observacion_docente,tareas_entregadas,tareas_tarde,participacion_nivel,nivel_riesgo,probabilidad_desercion,historial_cualitativo
0,RR19001,Carlos Ricardo,Ramírez Ramírez,1999-02-22,27,M,Ciencias Agronómicas,Ingeniería Agronómica,2019,10,...,6.15,5,0,Buen desempeno,3,2,Alta,Bajo,0.5760,"{'total_observaciones': 2, 'requiere_seguimien..."
1,RF24002,DANIELA CLAUDIA,Romero Flores,2006-03-23,20,FEMENINO,Ciencias Agronómicas,Licenciatura en Medicina Veterinaria,2024,6,...,6.99,0,0,Debe mejorar,2,1,Alta,Alto,0.1749,"{'total_observaciones': 2, 'requiere_seguimien..."
2,FG23003,mariana fatima,FLORES GONZÁLEZ,2004-10-01,21,FEMENINO,Ingeniería y Arquitectura,Ingeniería Eléctrica,2023,7,...,9.02,5,0,Requiere seguimiento,5,1,Baja,Bajo,0.2405,"{'total_observaciones': 1, 'requiere_seguimien..."
3,RL21004,roberto luis,Ramírez López,2002-08-21,24,M,Ciencias Agronómicas,Licenciatura en Medicina Veterinaria,2021,10,...,6.86,5,0,Buen desempeno,9,3,Alta,Medio,0.1089,"{'total_observaciones': 2, 'requiere_seguimien..."
4,PM21005,Hector Jorge,Pérez Morales,2002-11-24,23,m,Ciencias Agronómicas,Ingeniería Agronómica,2021,10,...,6.94,5,0,Sin observaciones,8,0,Baja,Medio,0.5903,"{'total_observaciones': 1, 'requiere_seguimien..."


In [ ]:
# Guardado del dataset de la parte 1, que es mi dataset integrado (esto para trabajar la parte 2, en la cual se hara la limpieza de este dataset)
df_consolidado.to_csv(DATA_PROCESSED/"dataset_consolidado_Proyecto_EdTech_Learning_Analytics.csv", index=False, encoding="utf-8")
print(f"Dataset final guardado en: {DATA_PROCESSED/'dataset_consolidado_Proyecto_EdTech_Learning_Analytics.csv'}")

print("Archivos generados")
print("datos/crudos: ")
!ls -la datos/crudos/

print("datos/externos: ")
!ls -la datos/externos/

print("datos/procesados: ")
!ls -la datos/procesados/

Dataset final guardado en: /content/datos/procesados/dataset_consolidado_Proyecto_EdTech_Learning_Analytics.csv
Archivos generados
datos/crudos: 
total 436
drwxr-xr-x 2 root root   4096 Sep 16 03:36 .
drwxr-xr-x 5 root root   4096 Sep 16 03:33 ..
-rw-r--r-- 1 root root 209709 Sep 16 03:35 Alumnado_UES.csv
-rw-r--r-- 1 root root 110592 Sep 16 03:36 Rendimiento_academicoUES.db
-rw-r--r-- 1 root root 112860 Sep 16 03:35 Rendimiento_academicoUES.sql
datos/externos: 
total 1084
drwxr-xr-x 2 root root    4096 Sep 16 03:36 .
drwxr-xr-x 5 root root    4096 Sep 16 03:33 ..
-rw-r--r-- 1 root root 1098282 Sep 16 03:36 bitacora_estudiantes_ues.json
datos/procesados: 
total 1084
drwxr-xr-x 2 root root    4096 Sep 16 03:37 .
drwxr-xr-x 5 root root    4096 Sep 16 03:33 ..
-rw-r--r-- 1 root root 1100986 Sep 16 03:37 dataset_consolidado_Proyecto_EdTech_Learning_Analytics.csv


**Avances realizados en presente Libro Version 1.0**

**Ultimo Autor: Luis Alonso Hernandez Mundo**

**-Creacion y Revision de carpeteo**

**-Carga de fuentes de datos CSV, SQL, JSON**

**-Exploracion de estrutura de fuentes de datos extraidas**

**-Consolidacion de datos creado para la unificacion de los dataset**

In [ ]:
# Inventariado de Fuentes
def obtener_detalle_fuente(df, nombre_fuente, tipo_archivo, descripcion):
    return {
        "Fuente": nombre_fuente,
        "Formato": tipo_archivo,
        "Filas": df.shape[0],
        "Columnas": df.shape[1],
        "Nombres de Variables": ", ".join(df.columns.tolist()),
        "Descripción": descripcion
    }

# Generacion de una lista detallada
resumen_fuentes = [
    obtener_detalle_fuente(df_Alumnado_UEScsv, "Alumnado_UES", "CSV", "Datos demográficos y socioeconómicos"),
    obtener_detalle_fuente(df_BaseRendimientoAcademicodb, "Rendimiento_academicoUES", "SQLite", "Notas y métricas académicas"),
    obtener_detalle_fuente(df_BitacoraEstudiantesjson, "bitacora_estudiantes_ues", "JSON", "Historial cualitativo")
]

df_inventario_detallado = pd.DataFrame(resumen_fuentes)

# Exportacion de nuestro inventariado de fuentes a un archivo CSV para anexar al informe
df_inventario_detallado.to_csv("/content/datos/procesados/inventario_de_fuentes.csv", index=False)
print("La inventario de fuente de datos se guardo exitosamente!")
display(df_inventario_detallado)

La inventario de fuente de datos se guardo exitosamente!


,Fuente,Formato,Filas,Columnas,Nombres de Variables,Descripción
0,Alumnado_UES,CSV,1000,19,"carnet, nombres, apellidos, fecha_nacimiento, ...",Datos demográficos y socioeconómicos
1,Rendimiento_academicoUES,SQLite,1000,14,"carnet, ciclo, materia, nota_teoria, nota_labo...",Notas y métricas académicas
2,bitacora_estudiantes_ues,JSON,1000,2,"carnet, historial_cualitativo",Historial cualitativo


In [ ]:
# CÁLCULO DE INDICADORES INICIALES

print("=========================================")
print("   INDICADORES INICIALES DEL PROYECTO    ")
print("=========================================\n")

# --- INDICADOR 1: Promedio Académico General ---
promedio_general = df_consolidado['nota_final'].mean().round(2)
print(f"1. PROMEDIO ACADÉMICO GENERAL: {promedio_general}")

# Desglose por facultad
promedio_facultad = df_consolidado.groupby('facultad')['nota_final'].mean().round(2)
print("\n   Desglose de Promedio Académico por Facultad:")
print(promedio_facultad.to_string())
print("-" * 50)

# --- INDICADOR 2: Asistencia (Faltas) ---
promedio_faltas = df_consolidado['faltas_consecutivas'].mean().round(2)
estudiantes_faltas_graves = df_consolidado[df_consolidado['faltas_consecutivas'] >= 5].shape[0]

print(f"\n2. ASISTENCIA (FALTAS CONSECUTIVAS):")
print(f"   - Promedio de faltas consecutivas: {promedio_faltas}")
print(f"   - Cantidad de estudiantes con 5 o más faltas seguidas: {estudiantes_faltas_graves}")
print("-" * 50)

# --- INDICADOR 3: Participación Virtual (Tareas y Nivel de Participación) ---
promedio_tareas = df_consolidado['tareas_entregadas'].mean().round(2)
conteo_participacion = df_consolidado['participacion_nivel'].value_counts()

print(f"\n3. PARTICIPACIÓN VIRTUAL:")
print(f"   - Promedio de tareas entregadas: {promedio_tareas}")
print("   - Distribución de niveles de participación:")
print(conteo_participacion.to_string())
print("=========================================")

   INDICADORES INICIALES DEL PROYECTO    

1. PROMEDIO ACADÉMICO GENERAL: 7.06

   Desglose de Promedio Académico por Facultad:
facultad
Ciencias Agronómicas                  6.94
Ciencias Económicas                   7.08
Ciencias Naturales y Matemática       7.00
Humanidades                           7.02
Ingeniería y Arquitectura             7.13
Jurisprudencia y Ciencias Sociales    6.97
Medicina                              7.30
--------------------------------------------------

2. ASISTENCIA (FALTAS CONSECUTIVAS):
   - Promedio de faltas consecutivas: 2.39
   - Cantidad de estudiantes con 5 o más faltas seguidas: 156
--------------------------------------------------

3. PARTICIPACIÓN VIRTUAL:
   - Promedio de tareas entregadas: 4.84
   - Distribución de niveles de participación:
participacion_nivel
Baja     352
Media    333
Alta     315


**Actualizaciones realizadas en el presente libro Version 1.1**

**Ultimos Autores: Luis Alonso Hernandez Mundo, Isela Alexandra Quintanilla Cruz**

**-Creacion del Inventariado de fuentes**

**-Creacion del calculo de indicadores iniciales**